In [0]:
from pyspark.sql import functions as F

In [0]:
print('Initializing Gold Layer building ... ')

df_orders = spark.read.table('de.p1.silver_orders')
df_products = spark.read.table('de.p1.bronze_products')
df_users_scd = spark.read.table('de.p1.silver_users_scd2').filter(F.col('is_current') == True)


Initializing Gold Layer building ... 


In [0]:
df_gold_fact = df_orders.alias('o') \
    .join(df_products.alias('p'), F.col('o.product_id') == F.col('p.product_id'), 'inner') \
    .join(df_users_scd.alias('u'), F.col('o.user_id') == F.col('u.user_id'), 'inner') \
    .select(
        F.col('o.order_id'),
        F.col('o.order_timestamp'),
        F.col('o.source_file_date').alias('order_date'),
        F.col('o.user_id'),
        F.col('u.tier').alias('user_tier'),
        F.col('u.city').alias('user_city'),
        F.col('o.product_id'),
        F.col('p.product_name'),
        F.col('p.category').alias('product_category'),
        F.col('o.quantity'),
        F.col('p.price'),
        (F.col('o.quantity') * F.col('p.price')).alias('total_amount'),
        F.col('o.order_status')
    )

df_gold_fact.write.format('delta').mode('overwrite').saveAsTable('de.p1.gold_order_details_fact')
print('Gold Layer orders details table has been built successfully!')

Gold Layer order table has been built successfully!


In [0]:
df_daily_sales = df_gold_fact.groupBy('order_date', 'product_category', 'user_tier').agg(
    F.countDistinct('order_id').alias('total_orders'),
    F.sum('quantity').alias('total_quantity_sold'),
    F.round(F.sum('total_amount'), 2).alias('total_gmv'),
    F.round(F.avg('total_amount'), 2).alias('avg_order_value')
).orderBy(F.desc('order_date'), F.desc('total_gmv'))

df_daily_sales.write.format('delta').mode('overwrite').saveAsTable('de.p1.gold_daily_sales_summary')
print('Gold layer daily sales table has been built successfully!')

Gold layer daily sales table has been built successfully!


In [0]:
display(spark.sql('select * from de.p1.gold_daily_sales_summary limit 10'))

order_date,product_category,user_tier,total_orders,total_quantity_sold,total_gmv,avg_order_value
2026-10-24,Electronics,Bronze,6,18,21695.69,3615.95
2026-10-24,Books,Platinum,2,10,9372.8,4686.4
2026-10-24,Beauty,Gold,7,24,9363.46,1337.64
2026-10-24,Electronics,Gold,2,8,8949.0,4474.5
2026-10-24,Books,Silver,6,20,8562.05,1427.01
2026-10-24,Apparel,Gold,5,16,8023.35,1604.67
2026-10-24,Home & Living,Silver,4,11,6064.08,1516.02
2026-10-24,Electronics,Silver,2,5,5972.99,2986.5
2026-10-24,Electronics,Platinum,1,4,5280.44,5280.44
2026-10-24,Books,Gold,4,12,5135.2,1283.8
